
# TF‑IDF + XGBoost (GPU) — **Generic, Large & Imbalanced Dataset Ready**

**Updated:** 2025-10-13 03:04 UTC  

Train on **any** text classification dataset from either:
- a **Hugging Face dataset** (by name), or
- a **CSV** file (local path or URL).

Includes:
- Robust label encoding (0..K-1) + preserved `LABEL_NAMES`
- Stratified splits (train/val/test)
- Imbalance handling via `compute_class_weight`
- XGBoost ≥ 2.0 GPU config (`tree_method="hist", device="cuda"`)
- Optional sampling & HashingVectorizer for very large data

### For **bbc-text.csv**
Set:
- `DATA_SOURCE = "csv"`  
- `CSV_PATH = "https://storage.googleapis.com/dataset-uploader/bbc/bbc-text.csv"`  
- `TEXT_COL = "text"`, `LABEL_COL = "category"`  


In [ ]:

import os, sys, platform, numpy as np, pandas as pd
import sklearn, xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer, HashingVectorizer
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt
import itertools

print("Python:", sys.version.replace("\n"," "))
print("OS:", platform.platform())
print("NumPy:", np.__version__)
print("pandas:", pd.__version__)
print("scikit-learn:", sklearn.__version__)
print("xgboost:", xgb.__version__)


In [ ]:

# GPU check (optional)
try:
    import shutil, subprocess
    if shutil.which("nvidia-smi"):
        print(subprocess.check_output(["nvidia-smi", "-L"]).decode())
    else:
        print("nvidia-smi not found — GPU may be unavailable in this runtime.")
except Exception as e:
    print("nvidia-smi check error:", e)

# datasets install
try:
    from datasets import load_dataset, Dataset
except Exception:
    import sys
    !pip -q install datasets
    from datasets import load_dataset, Dataset


In [ ]:

# ── CONFIG ─────────────────────────────────────────────────────────────────────
DATA_SOURCE = "csv"          # "hf" or "csv"

HF_DATASET_ID = "ag_news"
HF_TEXT_COL   = "text"
HF_LABEL_COL  = "label"

CSV_PATH  = "https://storage.googleapis.com/dataset-uploader/bbc/bbc-text.csv"
CSV_SEP   = ","
CSV_HEADER = "infer"
TEXT_COL  = "text"
LABEL_COL = "category"

FRACTION    = 1.0
SAMPLE_SIZE = None

VAL_SIZE  = 0.10
TEST_SIZE = None

RANDOM_STATE = 42

USE_HASHING = False
TFIDF_KW = dict(
    stop_words="english",
    ngram_range=(1, 2),
    max_features=100_000,
    min_df=3,
    max_df=0.95,
    sublinear_tf=True,
    smooth_idf=True,
    dtype=np.float32,
)
HASH_KW = dict(
    n_features=2**20,
    alternate_sign=False,
    ngram_range=(1,2),
    norm="l2"
)

XGB_KW = dict(
    objective="multi:softprob",
    n_estimators=3000,
    learning_rate=0.03,
    max_depth=8,
    subsample=0.9,
    colsample_bytree=0.9,
    reg_lambda=1.0,
    reg_alpha=0.0,
    tree_method="hist",
    device="cuda",
    eval_metric="mlogloss",
    n_jobs=-1,
    random_state=RANDOM_STATE,
    early_stopping_rounds=75,
)

USE_CLASS_WEIGHTS = True


In [ ]:

from datasets import load_dataset

def load_any_dataset(
    data_source="csv",
    hf_id=None, hf_text="text", hf_label="label",
    csv_path=None, csv_sep=",", csv_header="infer", text_col="text", label_col="label",
    test_size=None, random_state=42
):
    if data_source == "hf":
        ds = load_dataset(hf_id)
        if "test" in ds:
            df_train = pd.DataFrame({hf_text: ds["train"][hf_text], hf_label: ds["train"][hf_label]})
            df_test  = pd.DataFrame({hf_text: ds["test"][hf_text],  hf_label: ds["test"][hf_label]})
        else:
            df_all = pd.DataFrame({hf_text: ds["train"][hf_text], hf_label: ds["train"][hf_label]})
            if test_size is None:
                test_size = 0.10
            df_train, df_test = train_test_split(df_all, test_size=test_size, stratify=df_all[hf_label], random_state=random_state)
        text_c, label_c = hf_text, hf_label

    elif data_source == "csv":
        df_all = pd.read_csv(csv_path, sep=csv_sep, header=csv_header)
        assert text_col in df_all.columns, f"{text_col} not in CSV columns: {df_all.columns.tolist()}"
        assert label_col in df_all.columns, f"{label_col} not in CSV columns: {df_all.columns.tolist()}"
        if test_size:
            df_train, df_test = train_test_split(df_all, test_size=test_size, stratify=df_all[label_col], random_state=random_state)
        else:
            df_train, df_test = df_all, None
        text_c, label_c = text_col, label_col
    else:
        raise ValueError("DATA_SOURCE must be 'hf' or 'csv'")

    # Label encoding
    le = LabelEncoder()
    if df_test is not None:
        y_all = pd.concat([df_train[label_c], df_test[label_c]], axis=0)
        le.fit(y_all)
    else:
        le.fit(df_train[label_c])

    X_all = df_train[text_c].astype(str).tolist()
    y_all = le.transform(df_train[label_c]).astype(np.int64)

    X_train, X_val, y_train, y_val = train_test_split(
        X_all, y_all, test_size=VAL_SIZE, stratify=y_all, random_state=random_state
    )

    if df_test is not None:
        X_test = df_test[text_c].astype(str).tolist()
        y_test = le.transform(df_test[label_c]).astype(np.int64)
    else:
        X_test, y_test = [], np.array([], dtype=np.int64)

    label_names = le.classes_.tolist()
    return X_train, y_train, X_val, y_val, X_test, y_test, label_names

if DATA_SOURCE == "hf":
    X_train_list, y_train_arr, X_val_list, y_val_arr, X_test_list, y_test_arr, LABEL_NAMES = load_any_dataset(
        data_source="hf",
        hf_id=HF_DATASET_ID, hf_text=HF_TEXT_COL, hf_label=HF_LABEL_COL,
        test_size=TEST_SIZE, random_state=RANDOM_STATE
    )
else:
    X_train_list, y_train_arr, X_val_list, y_val_arr, X_test_list, y_test_arr, LABEL_NAMES = load_any_dataset(
        data_source="csv",
        csv_path=CSV_PATH, csv_sep=CSV_SEP, csv_header=CSV_HEADER,
        text_col=TEXT_COL, label_col=LABEL_COL,
        test_size=TEST_SIZE, random_state=RANDOM_STATE
    )

print("Sizes → Train:", len(X_train_list), " Val:", len(X_val_list), " Test:", len(X_test_list))
print("Label range (train):", (int(np.min(y_train_arr)), int(np.max(y_train_arr))))
print("Labels:", LABEL_NAMES)


In [ ]:

# Optional sampling
def maybe_sample(X_list, y_arr, fraction=None, sample_size=None, seed=42):
    if sample_size is not None and sample_size < len(X_list):
        n = sample_size
    elif fraction is not None and 0 < fraction < 1.0:
        n = int(len(X_list) * fraction)
    else:
        return X_list, y_arr
    rng = np.random.default_rng(seed)
    idx = rng.choice(len(X_list), size=n, replace=False)
    X_sub = [X_list[i] for i in idx]
    y_sub = y_arr[idx]
    return X_sub, y_sub

X_train_list, y_train_arr = maybe_sample(X_train_list, y_train_arr, FRACTION, SAMPLE_SIZE, seed=RANDOM_STATE)
print("After sampling → Train:", len(X_train_list))


In [ ]:

import scipy.sparse as sp
if USE_HASHING:
    vect = HashingVectorizer(**HASH_KW)
    Xtr = vect.transform(X_train_list)
    Xva = vect.transform(X_val_list)
    Xte = vect.transform(X_test_list) if len(X_test_list) else sp.csr_matrix((0, HASH_KW["n_features"]))
    vocab_info = f"HashingVectorizer(n_features={HASH_KW['n_features']})"
else:
    vect = TfidfVectorizer(**TFIDF_KW)
    Xtr = vect.fit_transform(X_train_list)
    Xva = vect.transform(X_val_list)
    Xte = vect.transform(X_test_list) if len(X_test_list) else sp.csr_matrix((0, len(vect.get_feature_names_out())))
    vocab_info = f"TfidfVectorizer(max_features={TFIDF_KW['max_features']})"

print("Vectorizer:", vocab_info)
print("Shapes → Xtr:", Xtr.shape, " Xva:", Xva.shape, " Xte:", Xte.shape)


In [ ]:

from xgboost import XGBClassifier
classes = np.unique(y_train_arr)
num_class = int(classes.max() + 1)

sample_weights = None
if USE_CLASS_WEIGHTS:
    class_weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_train_arr)
    cw_map = {c: w for c, w in zip(classes, class_weights)}
    sample_weights = np.array([cw_map[c] for c in y_train_arr], dtype=np.float32)
    print("Class weights:", cw_map)

params = dict(XGB_KW)
params["num_class"] = num_class

print(f"Training XGBoost {xgb.__version__} (device={params.get('device')}, tree_method={params.get('tree_method')})…")
clf = XGBClassifier(**params)
clf.fit(
    Xtr, y_train_arr,
    sample_weight=sample_weights,
    eval_set=[(Xva, y_val_arr)],
    verbose=100,
)


In [ ]:

def plot_confusion_matrix(cm, classes, normalize=False, title='Confusion matrix'):
    if normalize:
        with np.errstate(all='ignore'):
            cm = cm.astype('float') / cm.sum(axis=1, keepdims=True)
            cm = np.nan_to_num(cm)
    plt.figure()
    plt.imshow(cm, interpolation='nearest')
    plt.title(title)
    plt.colorbar()
    tick_marks = np.arange(len(classes))
    plt.xticks(tick_marks, classes, rotation=45, ha='right')
    plt.yticks(tick_marks, classes)
    fmt = '.2f' if normalize else 'd'
    for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
        plt.text(j, i, format(cm[i, j], fmt),
                 horizontalalignment="center", alpha=0.9)
    plt.ylabel('True label')
    plt.xlabel('Predicted label')
    plt.tight_layout()
    plt.show()

from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

target_names = LABEL_NAMES if len(LABEL_NAMES) == num_class else [str(i) for i in range(num_class)]

y_val_pred = clf.predict(Xva)
val_acc = accuracy_score(y_val_arr, y_val_pred)
print(f"Validation Accuracy: {val_acc:.4f}\n")
print("Validation Classification Report")
print(classification_report(y_val_arr, y_val_pred, target_names=target_names, digits=4, zero_division=0))

cm_val = confusion_matrix(y_val_arr, y_val_pred)
plot_confusion_matrix(cm_val, classes=target_names, normalize=True,
                      title="Validation Confusion Matrix (normalized)")

import os, joblib
SAVE_DIR = "tfidf_xgb_artifacts_generic"
os.makedirs(SAVE_DIR, exist_ok=True)
joblib.dump(clf, os.path.join(SAVE_DIR, "xgb_model.joblib"))
joblib.dump(vect, os.path.join(SAVE_DIR, "vectorizer.joblib"))
joblib.dump({"labels": target_names}, os.path.join(SAVE_DIR, "meta.joblib"))
print("Saved to:", SAVE_DIR)
